# Tutorial: `did_multiplegt_stat` in Python

## DiD estimators robust to heterogeneous effects

**Python implementation of the Stata package by de Chaisemartin & D'Haultfœuille (2024)**

---

## What is this package?

`did_multiplegt_stat` implements the **Difference-in-Differences** estimators proposed by **de Chaisemartin & D'Haultfœuille (2024)** for panels with:

- **Multiple groups** and **multiple time periods**
- **Non-binary treatments** (continuous or with multiple levels)
- **Staggered adoption**
- **Heterogeneous effects** across units and over time

### Why NOT use traditional TWFE?

Two-way fixed effects (TWFE) regression with heterogeneous effects can produce **biased** estimators — even **with the opposite sign of the true effect** (dCDH 2020). The estimators in this package are **robust to heterogeneity** and have a clear causal interpretation.

### Available estimators

| Estimator | Stata key | Python key | What it measures |
|-----------|-----------|------------|------------------|
| Average Slope | `as` | `aoss` | **Unweighted** average of the marginal effect of D |
| Weighted Average Slope | `was` | `waoss` | Average **weighted** by |ΔD| |
| IV-WAS | `ivwas` | `ivwaoss` | WAS with instrumental variable |

## 1. Setup — silence pandas warnings

The package emits some harmless `SettingWithCopyWarning` messages. We silence them with the canonical pandas option.

In [ ]:
import warnings, logging, os as _os
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')
for _cat in (FutureWarning, DeprecationWarning, UserWarning,
             RuntimeWarning, PendingDeprecationWarning):
    warnings.filterwarnings('ignore', category=_cat)
logging.getLogger('py.warnings').setLevel(logging.ERROR)
_os.environ['PYTHONWARNINGS'] = 'ignore'

import sys, os
import numpy as np
import pandas as pd

# Definitively silence SettingWithCopyWarning
pd.options.mode.chained_assignment = None

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 6)

# Add the package directory to the path
package_dir = r'C:\Users\Usuario\Documents\GitHub\py_did_multiplegt_stat\stat_python'
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)

from did_multiplegt_stat import did_multiplegt_stat

# The package sets the 'Agg' backend on import — revert to Jupyter inline
import matplotlib
try:
    matplotlib.use('module://matplotlib_inline.backend_inline', force=True)
except Exception:
    pass
import matplotlib.pyplot as plt

print('Package loaded successfully')

## 2. Load the `gazoline` dataset

Classic dataset from the official vignette — gasoline consumption by US state.

| Variable | Description |
|----------|-------------|
| `lngca`  | Log per-capita gasoline consumption (**outcome Y**) |
| `id`     | State (**ID**) |
| `year`   | Year (**Time**) |
| `tau`    | Fuel tax (**D**, continuous treatment) |

In [ ]:
data_path = os.path.join(package_dir, 'gazoline_did_multiplegt_stat.dta')
df = pd.read_stata(data_path)

print(f'Shape:        {df.shape}')
print(f'N states:     {df["id"].nunique()}')
print(f'Year range:   {int(df["year"].min())}–{int(df["year"].max())}')
print()
print(df[['id', 'year', 'lngca', 'tau']].head())

In [3]:
print(df[['lngca', 'tau']].describe().round(4))

           lngca        tau
count  2064.0000  2064.0000
mean     -0.4311    25.1014
std       0.1528    12.5688
min      -1.0354     9.0000
25%      -0.5136    12.0000
50%      -0.4237    24.0000
75%      -0.3436    37.1000
max       0.1779    56.3000


## 3. The specification

Original Stata line:

```stata
did_multiplegt_stat lngca id year tau, or(1) estimator(as was) placebo(2) exact_match
```

### Translation to the Python API

| Stata | Python |
|-------|--------|
| `lngca id year tau` | `Y='lngca', ID='id', Time='year', D='tau'` |
| `or(1)`             | `order=1` |
| `estimator(as was)` | `estimator=['aoss','waoss']` |
| `placebo(2)`        | `placebo=2` |
| `exact_match`       | `exact_match=True` |

### Concepts

- **`exact_match`**: counterfactuals built via **exact matching** on the baseline value of `tau`. Avoids functional-form assumptions.
- **`placebo(2)`**: estimates the same model 2 periods **before** the change. If parallel trends hold, placebos should be ≈ 0.

## 4. Estimation

In [ ]:
# Also silence the package's internal print for clean output
import io, contextlib

_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    result = did_multiplegt_stat(
        df=df,
        Y='lngca',
        ID='id',
        Time='year',
        D='tau',
        order=1,
        estimator=['aoss', 'waoss'],
        placebo=2,
        exact_match=True,
    )

print('Estimation completed')
print(f"  N obs:    {result['results']['N']}")
print(f"  N pairs:  {result['results']['pairs']}")
print(f"  Method:   {result['results']['WAOSS Method']}")
print(f"  Order:    {result['results']['Polynomial Order']}")

## 5. Results — Main estimators (AS and WAS)

In [ ]:
res = result['results']

main_table = res['table'].loc[['AS', 'WAS']].copy()
main_table.columns = ['Coef.', 'SE', '95% CI lower', '95% CI upper', 'Switchers', 'Stayers']

# Add z-stat and p-value
from scipy.stats import norm
main_table.insert(2, 'z',       (main_table['Coef.'] / main_table['SE']).round(3))
main_table.insert(3, 'p-value', (2 * (1 - norm.cdf(np.abs(main_table['z'])))).round(4))

print('=' * 78)
print('  Main estimators  —  lngca id year tau, or(1) placebo(2) exact_match')
print('=' * 78)
print(main_table.to_string())
print('=' * 78)

### Interpretation

- A **1-unit** increase in `tau` **reduces** log per-capita consumption by **~0.6% (AS)** and **~0.4% (WAS)**.
- Both 95% CIs **do not contain zero** → effect is **statistically significant**.
- The negative sign is consistent with economic theory: higher taxes → lower consumption.
- AS ≠ WAS because AS is a simple average over pairs, while WAS weights by |ΔD|.

## 6. Placebos — parallel trends test

In [ ]:
rows = []
for lag, table_p in [(1, res['table_placebo_1']), (2, res['table_placebo_2'])]:
    for est_label, row_key in [('AS', f'Placebo_{lag}'),
                                ('WAS', f'Placebo_{lag}_waoss')]:
        if row_key in table_p.index:
            r = table_p.loc[row_key]
            ci_lo = r['LB CI']
            ci_hi = r['UB CI']
            zero_in_ci = (ci_lo <= 0 <= ci_hi)
            rows.append({
                'Lag':           f't-{lag}',
                'Estimator':     est_label,
                'Coef.':         round(r['Estimate'], 6),
                'SE':            round(r['SE'],       6),
                '95% CI lower':  round(ci_lo,         6),
                '95% CI upper':  round(ci_hi,         6),
                '0 in CI':       'yes (OK)' if zero_in_ci else 'NO',
            })

placebos_df = pd.DataFrame(rows)
print('=' * 78)
print('  Placebos  —  4 placebos should contain 0 if parallel trends hold')
print('=' * 78)
print(placebos_df.to_string(index=False))
print('=' * 78)

n_ok = (placebos_df['0 in CI'] == 'yes (OK)').sum()
print(f"\n  {n_ok}/{len(placebos_df)} placebos contain 0 in their 95% CI "
      f"→ {'parallel trends OK' if n_ok == len(placebos_df) else 'WARNING'}")

## 7. Visualization — Event Study

In [ ]:
import io
from IPython.display import Image, display
import matplotlib.pyplot as plt

def plot_event_study(res, estimator='WAS', color='#1f77b4'):
    """Event study: 2 placebos + main effect."""
    suffix = '_waoss' if estimator == 'WAS' else ''
    p1 = res['table_placebo_1'].loc[f'Placebo_1{suffix}']
    p2 = res['table_placebo_2'].loc[f'Placebo_2{suffix}']
    m  = res['table'].loc[estimator]

    points = [(-2, p2['Estimate'], p2['SE']),
              (-1, p1['Estimate'], p1['SE']),
              ( 1, m['Estimate'],  m['SE'])]

    t  = np.array([p[0] for p in points])
    y  = np.array([p[1] for p in points])
    se = np.array([p[2] for p in points])

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.errorbar(t, y, yerr=1.96*se, fmt='o', markersize=11,
                color=color, ecolor=color, elinewidth=2,
                capsize=7, capthick=2, label=estimator)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(0, color='red',  linestyle=':', linewidth=1.5, alpha=0.6,
               label='D change')
    ax.set_xticks([-2, -1, 0, 1])
    ax.set_xticklabels(['Plac t-2', 'Plac t-1', '0', 'Effect t+1'])
    ax.set_xlabel('Time relative to treatment change', fontsize=12)
    ax.set_ylabel('Effect on log(per-capita consumption)', fontsize=12)
    ax.set_title(f'Event Study — {estimator} with placebos (95% CI)',
                 fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', frameon=True)
    plt.tight_layout()

    # Serialize the figure to PNG and display it as an embedded image
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue(), format='png'))

plot_event_study(res, estimator='AS',  color='#d62728')
plot_event_study(res, estimator='WAS', color='#1f77b4')

## 8. Equivalencia Stata ↔ Python

| Concepto            | Stata                               | Python                              |
|:--------------------|:------------------------------------|:------------------------------------|
| Comando             | `did_multiplegt_stat Y id year D`   | `did_multiplegt_stat(df, Y, ID, Time, D)` |
| Orden polinomio     | `, or(1)`                           | `order=1`                           |
| Estimadores         | `estimator(as was)`                 | `estimator=['aoss', 'waoss']`       |
| Placebos            | `placebo(2)`                        | `placebo=2`                         |
| Exact match         | `exact_match`                       | `exact_match=True`                  |
| Controles           | `controls(x1 x2)`                   | `controls=['x1', 'x2']`             |
| Cluster             | `cluster(state)`                    | `cluster='state'`                   |
| Peso                | `[pweight=w]`                       | `weight='w'`                        |
| Bootstrap           | `bootstrap(200)`                    | `bootstrap=200`                     |
| TWFE comparison     | `twfe`                              | `twfe=True`                         |
| Solo switchers up   | `switchers(up)`                     | `switchers='up'`                    |
| IV                  | `... Z, estimator(ivwas)`           | `Z='z', estimator='ivwaoss'`        |

## 9. References

- **de Chaisemartin, C. & D'Haultfœuille, X. (2020).** "Two-way fixed effects estimators with heterogeneous treatment effects." *American Economic Review*, 110(9), 2964–96.
- **de Chaisemartin, C. & D'Haultfœuille, X. (2024).** "Difference-in-Differences Estimators of Intertemporal Treatment Effects." *Review of Economics and Statistics*, forthcoming.
- Stata package: `ssc install did_multiplegt_stat`